# GPT-5.5 Agentic Capabilities — the Agents SDK

This notebook covers the **OpenAI Agents SDK**: the typed, server-side framework where *your* code owns orchestration, tool execution, state, and approvals. Recommended agent model: **`gpt-5.5`**.

We cover the primitives in dependency order: **Agent -> Runner -> Tools -> Handoffs -> Guardrails -> Tracing.**

> **Note on the older hosted Agent Builder:** the visual Agent Builder was deprecated (announced Jun 3 2026, shuts down Nov 30 2026). The migration path is **this SDK**. Don't teach Agent Builder.

> ## ⚠️ TODO — VERIFY IMPORT PATHS FIRST (prominent)
>
> The Agents SDK import surface is **[unverified]** and evolves between releases. **Before running anything below**, confirm the installed version and its actual import names:
>
> ```bash
> pip show openai-agents      # check installed version
> pip install -U openai-agents
> ```
>
> Then check the quickstart for the real names. The snippets here use the documented surface at research time, but these are the **most likely to have drifted**:
> - `Guardrail(...)` constructor and the `@input_guardrail` / `@output_guardrail` decorators
> - the `Tool(...)` wrapper vs the `@function_tool` decorator
> - `Runner.run_streamed()` vs `Runner.run_sync()` / `Runner.run()`
>
> If an import fails, fix it against the installed package rather than trusting these cells verbatim.

## Install

In [ ]:
# Install the Agents SDK ([unverified] package name -- confirm before teaching)
!pip install -U openai-agents

## 1. Agent — the LLM config (instructions + tools)

In [ ]:
from agents import Agent  # [unverified] import path

assistant = Agent(
    name="Assistant",
    model="gpt-5.5",
    instructions="Help users with their tasks. Be concise and pin your reasoning to evidence.",
)

## 2. Runner — execute the agent loop (streaming)

In [ ]:
import asyncio
from agents import Runner  # [unverified] import path

# Runner.run_streamed() returns a streaming run you can iterate event-by-event.
async def main():
    result = Runner.run_streamed(assistant, "Explain what an LLM agent is in 3 bullets.")
    async for event in result.stream_events():   # [unverified] event API
        print(event)
    print("\nFINAL:", result.final_output)

# In a notebook: await main()  (or asyncio.run(main()) in a script)
await main()

## 3. Tools — function tools + hosted tools (`shell`, `web_search`)

Function tools wrap your Python callables; hosted tools (`shell`, `web_search`) run on OpenAI's side.

In [ ]:
from agents import Agent, function_tool  # [unverified]

@function_tool
def add(x: int, y: int) -> int:
    """Add two integers."""
    return x + y

# Hosted tools: shell (container command execution) and web_search.
# [unverified] exact hosted-tool spec -- confirm names against the installed SDK.
research_agent = Agent(
    name="Researcher",
    model="gpt-5.5",
    instructions="Research questions using web search; compute with the add tool when needed.",
    tools=[
        add,
        {"type": "web_search"},   # hosted web search
        {"type": "shell"},        # hosted shell / container command execution
    ],
)

result = Runner.run_streamed(research_agent, "What's 21 + 21, and what's a recent AI release?")
async for event in result.stream_events():
    pass
print(result.final_output)

## 4. Handoffs — delegate to a specialist agent

A triage agent routes work to specialists; you control which agent ultimately replies.

In [ ]:
from agents import Agent, Runner  # [unverified]

refunds_agent = Agent(
    name="Refunds Specialist",
    model="gpt-5.5",
    instructions="Handle refund requests. Confirm order id, then explain the refund timeline.",
)

billing_agent = Agent(
    name="Billing Specialist",
    model="gpt-5.5",
    instructions="Answer billing and invoice questions precisely.",
)

triage_agent = Agent(
    name="Triage",
    model="gpt-5.5",
    instructions="Route the user to the right specialist. Hand off; do not answer directly.",
    handoffs=[refunds_agent, billing_agent],   # [unverified] handoffs API
)

result = Runner.run_streamed(triage_agent, "I want a refund for order #A1043.")
async for _ in result.stream_events():
    pass
print(result.final_output)

## 5. Input Guardrail — validate/block before the agent runs

Input guardrails inspect the user input and can block or pause risky requests (e.g. add a human-review checkpoint).

In [ ]:
# [unverified] guardrail API -- the decorator/return shape is the MOST likely to have drifted.
from agents import Agent, Runner, input_guardrail, GuardrailFunctionOutput

@input_guardrail
async def block_secrets(ctx, agent, user_input: str) -> GuardrailFunctionOutput:
    """Block prompts that look like they contain credentials."""
    tripwire = "api_key" in user_input.lower() or "password" in user_input.lower()
    return GuardrailFunctionOutput(
        output_info={"reason": "looks like a secret"} if tripwire else {},
        tripwire_triggered=tripwire,
    )

guarded_agent = Agent(
    name="Guarded Assistant",
    model="gpt-5.5",
    instructions="Help with safe requests only.",
    input_guardrails=[block_secrets],   # [unverified]
)

# This should trip the input guardrail:
try:
    result = Runner.run_streamed(guarded_agent, "Here is my password: hunter2, log me in.")
    async for _ in result.stream_events():
        pass
    print(result.final_output)
except Exception as e:
    print("Input guardrail blocked the run:", type(e).__name__, e)

## 6. Output Guardrail — validate the agent's answer before returning

Output guardrails inspect the produced answer (e.g. ensure no PII / no overconfident claims) and can block it.

In [ ]:
# [unverified] output guardrail API
from agents import Agent, Runner, output_guardrail, GuardrailFunctionOutput

@output_guardrail
async def no_absolute_promises(ctx, agent, agent_output: str) -> GuardrailFunctionOutput:
    """Block answers that make absolute guarantees."""
    banned = ("guaranteed", "100% safe", "never fails")
    hit = any(b in str(agent_output).lower() for b in banned)
    return GuardrailFunctionOutput(
        output_info={"reason": "absolute promise"} if hit else {},
        tripwire_triggered=hit,
    )

careful_agent = Agent(
    name="Careful Assistant",
    model="gpt-5.5",
    instructions="Answer helpfully but never make absolute guarantees.",
    output_guardrails=[no_absolute_promises],   # [unverified]
)

try:
    result = Runner.run_streamed(careful_agent, "Is this investment safe?")
    async for _ in result.stream_events():
        pass
    print(result.final_output)
except Exception as e:
    print("Output guardrail blocked the answer:", type(e).__name__, e)

## 7. Tracing — view, debug, and optimize runs

The SDK records traces of every agent run. Wrap work in a `trace(...)` context to group it.

**Opening the trace viewer:** traces are sent to the OpenAI platform — open the **Traces** dashboard at **https://platform.openai.com/traces** to inspect each run (agent steps, tool calls, handoffs, guardrail trips). From there you can progress from traces to eval loops. *(Confirm the exact dashboard path in your account — the hosted Evals platform is being deprecated, so prefer the SDK tracing -> code-based eval workflow.)*

In [ ]:
from agents import trace, Runner  # [unverified]

with trace("course-agentic-demo"):
    result = Runner.run_streamed(assistant, "Summarize this notebook's agent primitives in 5 bullets.")
    async for _ in result.stream_events():
        pass
    print(result.final_output)

print("\nOpen https://platform.openai.com/traces to inspect the 'course-agentic-demo' trace.")

## Appendix — hosted tools via the plain Responses API (no SDK)

You don't always need the Agents SDK. Hosted tools like `web_search` also work directly on `client.responses.create()`. This is the lighter-weight path for single-shot tool use.

In [ ]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5.5",
    tools=[{"type": "web_search"}],
    input="What are the latest news on AI tool releases? Give me a top-5 list.",
    reasoning={"effort": "medium"},
)
print(response.output_text)